# Early-warning training and evaluation (GPU server)

This notebook lives in `notebooks/`. It finds the repository root (`src/`, `configs/`, `scripts/`, and `data/raw/`) from the working directory. The Chronos caches are already on this server.

Restart the kernel, then Run All.

XGBoost fits on CUDA (`tree_method` stays `hist`; the frozen grid is unchanged). Logistic regression, calibration, and the bootstrap use CPUs in this same job. Chronos is not run again.

In [ ]:
from pathlib import Path
import os
import shutil

def repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        marker = candidate / "scripts" / "run_early_warning_experiment.py"
        if marker.exists():
            return candidate
    raise SystemExit(f"repository root not found from {start}")

ROOT = repo_root()
print("root:", ROOT)
CHRONOS_SRC = Path("/net/home/rfozdar/ERCOT Paper Test/chronos")
PKG = Path("/tmp/ew_pkg")

needed = [
    ROOT / "src" / "experiment.py",
    ROOT / "configs" / "experiment_early_warning.yaml",
    ROOT / "scripts" / "run_early_warning_experiment.py",
    ROOT / "data" / "raw" / "ercot_rt_prices.parquet",
    ROOT / "data" / "raw" / "ercot_da_prices.parquet",
    ROOT / "data" / "raw" / "ercot_load.parquet",
    ROOT / "data" / "raw" / "weather.parquet",
    ROOT / "data" / "raw" / "weather_forecast_gfs.parquet",
    ROOT / "data" / "raw" / "gdelt.parquet",
    ROOT / "data" / "raw" / "eia_gas.parquet",
    ROOT / "data" / "raw" / "eia_storage.parquet",
    ROOT / "data" / "raw" / "nrc_reactors.parquet",
]
missing = [str(p) for p in needed if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Upload the repository, including data/raw, and open this notebook from its root.\n"
        + "\n".join(missing)
    )
print("repo:", ROOT)

In [ ]:
import subprocess
import sys

PKG.mkdir(exist_ok=True)
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--target", str(PKG), "--upgrade",
    "xgboost>=2.0", "scikit-learn", "scipy", "pyyaml", "matplotlib", "pyarrow",
])
sys.path.insert(0, str(PKG))

import torch
import xgboost as xgb

assert torch.cuda.is_available(), "CUDA is not available in this kernel."
info = xgb.build_info()
assert info.get("USE_CUDA"), f"This xgboost build has no CUDA: {info}"
print(f"xgboost {xgb.__version__}  torch {torch.__version__}  {torch.cuda.get_device_name(0)}")
print("xgboost loaded from", Path(xgb.__file__).resolve())

In [ ]:
dest_dir = ROOT / "data" / "features" / "chronos"
dest_dir.mkdir(parents=True, exist_ok=True)
caches = [
    "bolt-small_ctx512.npz",
    "bolt-small_ctx256.npz",
    "bolt-small_ctx1024.npz",
    "bolt-base_ctx512.npz",
    "t5-small_ctx512.npz",
]
for name in caches:
    src = CHRONOS_SRC / name
    dest = dest_dir / name
    if not src.exists() and not dest.exists():
        raise FileNotFoundError(src)
    if src.exists() and (not dest.exists() or src.stat().st_size != dest.stat().st_size):
        shutil.copy2(src, dest)
        print("copied", name)
    else:
        print("have", dest.name, f"({dest.stat().st_size / 1e6:.1f} MB)")

In [ ]:
env = os.environ.copy()
env["ERCOT_XGB_DEVICE"] = "cuda"
env["PYTHONPATH"] = str(PKG) + os.pathsep + str(ROOT) + os.pathsep + env.get("PYTHONPATH", "")
subprocess.check_call(
    [sys.executable, "-u", "scripts/run_early_warning_experiment.py"],
    cwd=ROOT, env=env,
)
print("done. Download reports/results/ from the repo.")